<div style="border-top:4px solid #0f766e;padding:28px 0 18px">
<div style="color:#0f766e;font-size:13px;font-weight:700;letter-spacing:.8px">LAB 10 · LEVEL 3 · MANAGING DATA</div>
<div style="color:#17212b;font-size:30px;font-weight:750">Manage partitions with evidence and boundaries</div>
<p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px">Inspect the physical layout of an isolated table, add a lifecycle partition, remove one partition safely, and explain why logical data management is separate from background compaction.</p>
<span style="display:inline-block;border:1px solid #99f6e4;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:10px;font-size:12px">Use an isolated sandbox · never manage the shared Level 1 baseline</span>
</div>

## Management boundary

Management actions are intentionally destructive. This lab owns only `management_events_lab10` and `management_scratch_lab10`; it never truncates, alters, or drops `events` or another module's table. Run the cells in order and read the metadata result before changing data.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")


In [ ]:
lab.execute('''
CREATE TABLE IF NOT EXISTS management_events_lab10 (
    event_date DATE NOT NULL,
    event_id BIGINT NOT NULL,
    event_type VARCHAR(20) NOT NULL,
    revenue DECIMAL(12,2) NOT NULL
)
DUPLICATE KEY(event_date, event_id)
PARTITION BY RANGE(event_date) (
    PARTITION p202001 VALUES LESS THAN ("2020-02-01"),
    PARTITION p202002 VALUES LESS THAN ("2020-03-01")
)
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
''')
lab.execute("TRUNCATE TABLE management_events_lab10")
lab.insert('''
INSERT INTO management_events_lab10 VALUES
    ("2020-01-15", 100001, "purchase", 29.95),
    ("2020-01-20", 100002, "view", 0.00),
    ("2020-02-05", 100003, "purchase", 49.00),
    ("2020-02-06", 100004, "cart", 0.00)
''', title="Seed the isolated management table")

## 1. Inspect before changing

`SHOW CREATE TABLE` is the design contract. `SHOW PARTITIONS` reports lifecycle state and row counts. `SHOW TABLETS` is physical evidence about the tablets owned by the table.

In [ ]:
lab.sql(
    "SHOW CREATE TABLE management_events_lab10",
    title="Management table definition",
)
lab.sql(
    "SHOW PARTITIONS FROM management_events_lab10 ORDER BY PartitionName",
    title="Partition lifecycle before maintenance",
    columns=["PartitionName", "State", "Buckets", "ReplicationNum", "RowCount"],
)
lab.sql(
    "SHOW TABLETS FROM management_events_lab10",
    title="Tablet layout before maintenance",
    columns=["TabletId", "PartitionId", "BackendId", "State", "LstSuccessVersion"],
)

## 2. Add the next lifecycle partition

Adding a partition makes the retention boundary explicit. It does not move existing rows or rewrite another partition.

In [ ]:
lab.execute('ALTER TABLE management_events_lab10 ADD PARTITION p202003 VALUES LESS THAN ("2020-04-01")')
lab.insert('''
INSERT INTO management_events_lab10 VALUES
    ("2020-03-03", 100005, "purchase", 12.50)
''', title="Load the new lifecycle partition")
lab.sql(
    "SELECT event_date, COUNT(*) AS rows_in_partition, SUM(revenue) AS revenue FROM management_events_lab10 GROUP BY event_date ORDER BY event_date",
    title="Rows across the lifecycle boundary",
)

## 3. Truncate one partition, not the table

This operation models retention of January data. The February and March partitions must remain queryable. A partition-level operation is safer than `TRUNCATE TABLE` when the business boundary is time-based.

In [ ]:
lab.execute('TRUNCATE TABLE management_events_lab10 PARTITION (p202001)')
lab.sql(
    "SELECT event_date, COUNT(*) AS row_count, SUM(revenue) AS revenue FROM management_events_lab10 GROUP BY event_date ORDER BY event_date",
    title="Data after January partition retention",
)
lab.sql(
    "SHOW PARTITIONS FROM management_events_lab10 ORDER BY PartitionName",
    title="Partition metadata after retention",
    columns=["PartitionName", "State", "Buckets", "ReplicationNum", "RowCount"],
    final=True,
)

## 4. Separate logical maintenance from physical reclamation

A partition can be logically empty while old rowsets and segment files wait for normal background cleanup. The metadata query is the evidence for the operation; do not claim that a notebook query proves immediate disk reclamation.

In [ ]:
lab.execute('''
CREATE TABLE IF NOT EXISTS management_scratch_lab10 (
    id BIGINT NOT NULL,
    note VARCHAR(32) NOT NULL
)
DUPLICATE KEY(id)
DISTRIBUTED BY HASH(id) BUCKETS 1
PROPERTIES ("replication_num"="1")
''')
lab.execute("TRUNCATE TABLE management_scratch_lab10")
lab.insert("INSERT INTO management_scratch_lab10 VALUES (1, 'temporary')", title="Create isolated scratch state")
lab.sql("SELECT * FROM management_scratch_lab10 ORDER BY id", title="Scratch table before cleanup")
lab.execute("TRUNCATE TABLE management_scratch_lab10")
lab.sql("SELECT COUNT(*) AS remaining_rows FROM management_scratch_lab10", title="Scratch table after explicit cleanup", final=True)

## Takeaway

- Inspect DDL and partition metadata before a destructive operation.
- Prefer partition-level lifecycle operations when the retention boundary is a partition.
- `SHOW TABLETS` provides layout evidence; it is not a substitute for a compaction or storage-health check.
- Logical visibility and physical reclamation happen at different times. Keep those claims separate in an operational runbook.